In [ ]:
!git clone https://github.com/kethansplunk/Codegen.git
%cd Codegen
!pip install -q torch transformers peft sqlparse pyyaml FlagEmbedding chromadb openai python-dotenv


Cloning into 'Codegen'...
remote: Enumerating objects: 1092, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 1092 (delta 88), reused 102 (delta 42), pack-reused 931 (from 1)
Receiving objects: 100% (1092/1092), 17.65 MiB | 15.28 MiB/s, done.
Resolving deltas: 100% (807/807), done.
/content/Codegen
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 144.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 137.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 22.6 MB/s eta

KeyboardInterrupt: 

In [ ]:
# 1. Mount Drive — this is where Phase 12/13/14 saved checkpoints
from google.colab import drive
drive.mount('/content/drive')

# 2. Install deps (skip torch/transformers if the Colab image already has them)
!pip install -q peft sqlparse pyyaml FlagEmbedding chromadb openai python-dotenv

# 3. Symlink Drive checkpoints into the repo's expected relative paths
#    (configs/config.yaml expects models/... and indexes/... relative to repo root)
import os
%cd /content/Codegen

DRIVE = '/content/drive/MyDrive/codegen'
os.makedirs('models', exist_ok=True)
os.makedirs('indexes', exist_ok=True)

os.symlink(f'{DRIVE}/checkpoints/sar_sql',       'models/sar_sql')
os.symlink(f'{DRIVE}/checkpoints/generator_sql', 'models/generator_sql')
os.symlink(f'{DRIVE}/indexes/chroma_sql',        'indexes/chroma_sql')

# 4. Sanity check the symlinks actually resolve
!ls -la models/sar_sql models/generator_sql indexes/chroma_sql


Mounted at /content/drive
/content/Codegen
lrwxrwxrwx 1 root root 49 Jul 11 05:01 indexes/chroma_sql -> /content/drive/MyDrive/codegen/indexes/chroma_sql
lrwxrwxrwx 1 root root 56 Jul 11 05:01 models/generator_sql -> /content/drive/MyDrive/codegen/checkpoints/generator_sql
lrwxrwxrwx 1 root root 50 Jul 11 05:01 models/sar_sql -> /content/drive/MyDrive/codegen/checkpoints/sar_sql


In [ ]:
import os, shutil

# Remove the broken symlink (safe — only removes the link, not the Drive files)
if os.path.islink('indexes/chroma_sql'):
    os.remove('indexes/chroma_sql')
elif os.path.exists('indexes/chroma_sql'):
    shutil.rmtree('indexes/chroma_sql')

# Real local copy this time
shutil.copytree('/content/drive/MyDrive/codegen/indexes/chroma_sql', 'indexes/chroma_sql')

print(os.listdir('indexes/chroma_sql'))


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/codegen/indexes/chroma_sql'

Smoke test


In [ ]:
!python -m scripts.run_posg_sql --smoke_test --n 10


Running on: cuda
[warn] Data/Spider/database/ not found — EX comparison will be skipped, falling back to exact-string-match only.

Loading SAR retriever ...
config.json: 100% 779/779 [00:00<00:00, 4.97MB/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 2.51MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 73.9MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 145MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 920kB/s]
model.safetensors: 100% 1.34G/1.34G [00:03<00:00, 361MB/s] 
Loading weights: 100% 391/391 [00:00<00:00, 2879.94it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/Codegen/scripts/run_posg_sql.py", line 235, in <module>
    main()
  File "/content/Codegen/scripts/run_posg_sql.py", line 225, in main
    smoke_test(config, n=args.n, strategy=args.strategy, seed=args.seed)
  File "/content/Codegen/scripts/run_posg_sql.py", line 122, in smoke_test
    sar = 

In [ ]:
!ls -la models/sar_sql/ models/generator_sql/
!file models/sar_sql/sar_model.pt



models/generator_sql/:
total 642019
-rw------- 1 root root      1109 Jul  4 07:58 adapter_config.json
-rw------- 1 root root 645975704 Jul  4 07:58 adapter_model.safetensors
-rw------- 1 root root      2507 Jul  4 07:58 chat_template.jinja
drwx------ 2 root root      4096 Jul  4 07:58 checkpoint-1266
drwx------ 2 root root      4096 Jul  4 06:03 checkpoint-422
drwx------ 2 root root      4096 Jul  4 07:00 checkpoint-844
-rw------- 1 root root      5214 Jul  4 07:58 README.md
-rw------- 1 root root       690 Jul  4 07:58 tokenizer_config.json
-rw------- 1 root root  11421892 Jul  4 07:58 tokenizer.json
-rw------- 1 root root      5265 Jul  4 07:58 training_args.bin

models/sar_sql/:
total 49224
-rw------- 1 root root 50404891 Jun 29 06:03 sar_model.pt
models/sar_sql/sar_model.pt: Zip archive data, at least v0.0 to extract, compression method=store


In [ ]:
!python -m scripts.run_posg_sql --smoke_test --n 10


Running on: cuda
[warn] Data/Spider/database/ not found — EX comparison will be skipped, falling back to exact-string-match only.

Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 2993.33it/s]
Loaded corpus: 7000 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 28/28 [00:00<00:00, 87.15it/s]
Inference Embeddings: 100% 28/28 [00:00<00:00, 33.23it/s]
Loading Generator ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 88.25it/s] 

[1/10] Show the number of transactions for different investors.
  gold:     SELECT investor_id ,  COUNT(*) FROM TRANSACTIONS GROUP BY investor_id
  greedy:   SELECT investor_id ,  COUNT(*) FROM TRANSACTIONS GROUP BY investor_id
  selected: SELECT investor_id ,  COUNT(*) FROM TRANSACTIONS GROUP BY investor_id  [same as greedy]
  pareto_front_size=5  unique_candidates=1
  exact_match: posg=True greedy=True   |  EX skipped (no local Spider DB)

[2/10] What are the rent

In [ ]:
!python -m scripts.run_posg_sql --smoke_test --n 10 --hard


Running on: cuda
--hard: filtered 6748 -> 4159 multi-join/subquery/group-by entries
[warn] Data/Spider/database/ not found — EX comparison will be skipped, falling back to exact-string-match only.

Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 2942.91it/s]
Loaded corpus: 7000 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 28/28 [00:00<00:00, 85.14it/s]
Inference Embeddings: 100% 28/28 [00:00<00:00, 32.95it/s]
Loading Generator ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 87.83it/s] 

[1/10] What are the statement id and statement detail for the statement that has the most corresponding accounts?
  gold:     SELECT T1.statement_id ,  T2.statement_details FROM Accounts AS T1 JOIN Statements AS T2 ON T1.statement_id  =  T2.statement_id GROUP BY T1.statement_id ORDER BY count(*) DESC LIMIT 1
  greedy:   SELECT T1.statement_id ,  T2.statement_details FROM Accounts AS T1 JOIN Statement

In [ ]:
%cd /content/Codegen

/content/Codegen


In [ ]:
!python -m scripts.build_dev_eval_set --limit 30 --out Data/cot_data/sql_dev_eval_sample.json


Dev entries: 30
  20/30 processed (20 written, 0 skipped: no cached schema)
  30/30 processed (30 written, 0 skipped: no cached schema)

Done. 30 dev entries -> Data/cot_data/sql_dev_eval_sample.json  (0 skipped: schema not cached locally)


In [ ]:
!python -m scripts.run_posg_sql --smoke_test --n 10 --hard \
    --data Data/cot_data/sql_dev_eval_sample.json


Running on: cuda
Data source: Data/cot_data/sql_dev_eval_sample.json
--hard: filtered 30 -> 12 multi-join/subquery/group-by entries
[warn] Data/Spider/database/ not found — EX comparison will be skipped, falling back to exact-string-match only.

Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 2846.69it/s]
Loaded corpus: 7000 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 28/28 [00:00<00:00, 86.67it/s]
Inference Embeddings: 100% 28/28 [00:00<00:00, 33.15it/s]
Loading Generator ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 88.46it/s]

[1/10] Show the stadium name and capacity with most number of concerts in year 2014 or after.
  gold:     SELECT T2.name ,  T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T1.year  >=  2014 GROUP BY T2.stadium_id ORDER BY count(*) DESC LIMIT 1
  greedy:   SELECT T1.name ,  T1.capacity FROM stadium AS T1 JOIN con

In [ ]:
!python -m scripts.run_posg_sql --smoke_test --n 10 --hard \
    --data Data/cot_data/sql_dev_eval_sample.json


Running on: cuda
Data source: Data/cot_data/sql_dev_eval_sample.json
--hard: filtered 30 -> 12 multi-join/subquery/group-by entries
[warn] Data/Spider/database/ not found — EX comparison will be skipped, falling back to exact-string-match only.

Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 2695.03it/s]
Loaded corpus: 7000 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 28/28 [00:00<00:00, 86.21it/s]
Inference Embeddings: 100% 28/28 [00:00<00:00, 32.92it/s]
Loading Generator ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 88.46it/s] 

[1/10] Show the stadium name and capacity with most number of concerts in year 2014 or after.
  gold:     SELECT T2.name ,  T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T1.year  >=  2014 GROUP BY T2.stadium_id ORDER BY count(*) DESC LIMIT 1
  greedy:   SELECT T1.name ,  T1.capacity FROM stadium AS T1 JOIN co

In [ ]:
!cp /content/drive/MyDrive/codegen/checkpoints/spider_database.zip /content/Codegen/
!unzip -q /content/Codegen/spider_database.zip -d /content/Codegen/Data/Spider/
!ls /content/Codegen/Data/Spider/database | wc -l
!ls /content/Codegen/Data/Spider/database | head -5


166
academic
activity_1
aircraft
allergy_1
apartment_rentals


In [ ]:
!python -m scripts.run_posg_sql --smoke_test --n 10 --hard \
    --data Data/cot_data/sql_dev_eval_sample.json


Running on: cuda
Data source: Data/cot_data/sql_dev_eval_sample.json
--hard: filtered 30 -> 12 multi-join/subquery/group-by entries
Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 2827.25it/s]
Loaded corpus: 7000 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 28/28 [00:00<00:00, 86.30it/s]
Inference Embeddings: 100% 28/28 [00:00<00:00, 32.88it/s]
Loading Generator ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 88.32it/s] 

[1/10] Show the stadium name and capacity with most number of concerts in year 2014 or after.
  gold:     SELECT T2.name ,  T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T1.year  >=  2014 GROUP BY T2.stadium_id ORDER BY count(*) DESC LIMIT 1
  greedy:   SELECT T2.name ,  T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T1.year  >=  2014 GROUP BY T1.stadium_id ORDER BY count(*) DE

In [ ]:
!python -m scripts.build_dev_eval_set --out Data/cot_data/sql_dev_eval_full.json


Dev entries: 1034
  20/1034 processed (20 written, 0 skipped: no cached schema)
  40/1034 processed (40 written, 0 skipped: no cached schema)
  60/1034 processed (60 written, 0 skipped: no cached schema)
  80/1034 processed (80 written, 0 skipped: no cached schema)
  100/1034 processed (100 written, 0 skipped: no cached schema)
  120/1034 processed (120 written, 0 skipped: no cached schema)
  140/1034 processed (140 written, 0 skipped: no cached schema)
  160/1034 processed (160 written, 0 skipped: no cached schema)
  180/1034 processed (180 written, 0 skipped: no cached schema)
  200/1034 processed (200 written, 0 skipped: no cached schema)
  220/1034 processed (220 written, 0 skipped: no cached schema)
  240/1034 processed (240 written, 0 skipped: no cached schema)
  260/1034 processed (260 written, 0 skipped: no cached schema)
  280/1034 processed (280 written, 0 skipped: no cached schema)
  300/1034 processed (300 written, 0 skipped: no cached schema)
  320/1034 processed (320 writ

In [ ]:
!python -m scripts.run_posg_sql --smoke_test --n 30 --hard \
    --data Data/cot_data/sql_dev_eval_full.json


Running on: cuda
Data source: Data/cot_data/sql_dev_eval_full.json
--hard: filtered 1034 -> 606 multi-join/subquery/group-by entries
Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 2837.21it/s]
Loaded corpus: 7000 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 28/28 [00:00<00:00, 82.81it/s]
Inference Embeddings: 100% 28/28 [00:00<00:00, 32.14it/s]
Loading Generator ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 87.58it/s] 

[1/30] What are the names of people who do not play poker?
  gold:     SELECT Name FROM people WHERE People_ID NOT IN (SELECT People_ID FROM poker_player)
  greedy:   SELECT Name FROM people WHERE People_id NOT IN (SELECT People_id FROM poker_player)
  selected: SELECT Name FROM people WHERE People_id NOT IN (SELECT People_id FROM poker_player)  [same as greedy]
  pareto_front_size=5  unique_candidates=2
  exact_match: posg=True greedy=True   |  EX: posg=1.0 greed